In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.regularizers import l2

import sys
sys.path.append('../')
from Utils import utils_nn as utlnn

In [ ]:
R0= 6.371E6 # [m]
out_y_coord = [f'z_{i}' for i in range(1,201)]

x_test = pd.read_excel("../Train_Test/Dataset_Separado_xyz/x_test_new.xlsx")
x_train = pd.read_excel("../Train_Test/Dataset_Separado_xyz/x_train_new.xlsx")

y_test = pd.read_excel("../Train_Test/Dataset_Separado_xyz/y_test_new.xlsx",
                      usecols = out_y_coord)
y_train = pd.read_excel("../Train_Test/Dataset_Separado_xyz/y_train_new.xlsx",
                      usecols = out_y_coord)

print("Forma de x antes del drop:\n", x_test.head())
x_train	= x_train.drop(columns = ['mmdd_modified', 'hour', 'day_of_year'])
x_test	= x_test.drop(columns =[ 'mmdd_modified', 'hour', 'day_of_year'])
print("Forma de x post drop:\n", x_test.head())

In [ ]:
#Normalizamos la salida
from sklearn.preprocessing import MinMaxScaler, StandardScaler 
import joblib
# Entradas.
cols_to_scale = ['fc [Mhz]','elevation','azimuth']
scaler_inputs = StandardScaler()
scaler_inputs.fit(x_train[cols_to_scale])

x_train[cols_to_scale] = scaler_inputs.transform(x_train[cols_to_scale])
x_test[cols_to_scale] = scaler_inputs.transform(x_test[cols_to_scale])
# Salidas.
scaler_z = MinMaxScaler()
y_train_z_scaled = scaler_z.fit_transform(y_train)
y_test_z_scaled = scaler_z.transform(y_test)
# Guardamos scalers para futuras predicts 
joblib.dump(scaler_inputs, '../Scalers_Models/scaler_mayo/scalerZ/scaler_inputs.pkl')
joblib.dump(scaler_z, '../Scalers_Models/scaler_mayo/scalerZ/scaler_outputs_y.pkl')

In [ ]:
from tensorflow.keras.optimizers import Adam

early_stopping = EarlyStopping(
  monitor = 'val_loss',	#monitoriamos la pérdida en validación
  patience = 30, # Si no mejora en 10->20 epochs, detenemos el entrenamiento.
  restore_best_weights = True # Restaura los mejores pesos encontrados.
)
reduce_lr = ReduceLROnPlateau(
  monitor = 'val_loss',
  patience = 20,
  factor = 0.5
)
# Pre def ===========>>>>>>>
act_name = "relu"
l2_reg = 0.0  
epoch = 600
b_s = 32
optimizer_name = Adam(learning_rate = 1e-3)
# ===================>>>>>>>
def train_model(x_train, y_train_z_scaled):
  """
  Función para entrenar el modelo en coordenada Z.
  
  Arguments:
  - x_train: DataFrame de entrenamiento de entradas.
  - y_train_z_scaled: DataFrame de entrenamiento de salidas en coordenada Z.
  Returns:
  - model: Modelo entrenado.
  - history: Historial de entrenamiento del modelo.
  
  """
  inputs = Input(shape=(x_train.shape[1],))
  # Capa oculta 1
  x = Dense(300, activation=act_name, kernel_regularizer=l2(l2_reg))(inputs)
	# Capa oculta 2
  x = Dense(400, activation=act_name, kernel_regularizer=l2(l2_reg))(x)
  # Capa oculta 3
  x = Dense(400, activation=act_name, kernel_regularizer=l2(l2_reg))(x)
	# Capa de oculta 4
  x = Dense(300, activation=act_name, kernel_regularizer=l2(l2_reg))(x)
  # Capa de salida
  outputs = Dense(200, activation = 'linear', name = 'output_Z')(x)
  
  model = Model(inputs=inputs, outputs=outputs, name = 'altitude_model')
  model.compile(optimizer = optimizer_name, loss = 'mse')
  model.summary()
  history = model.fit(x_train, y_train_z_scaled,
                      epochs = epoch,
											batch_size = b_s,
											validation_split = 0.2,
											callbacks = [early_stopping, reduce_lr],
                      verbose = 1
											)
  return model, history
print("x_train.shape =", x_train.shape)
print("y_train_z_scaled.shape =", y_train_z_scaled.shape)

print("Tipo x_train:", type(x_train))
print("Tipo y_train_z_scaled:", type(y_train_z_scaled))
model_z, history_z = train_model(x_train,y_train_z_scaled)
  

In [ ]:
loss = model_z.evaluate(x_test,y_test_z_scaled)
print(f'Pérdida en datos de Test: {loss}')

In [ ]:
idx = 1235
# Elegir una muestra para comparar.

y_pred_scaled = model_z.predict(np.expand_dims(x_test.iloc[idx], axis=0))

y_true = y_test.iloc[idx] # Se obtine Algo de tipo Serie
y_true=y_true.to_numpy() # Transform a Numpy array

#Desnormalizamos
y_pred = scaler_z.inverse_transform(y_pred_scaled)
y_pred = y_pred.flatten() # [[...,...,...,....,]]


def graf_1D(y_true,y_pred):
  ##Graficamos 
  # Buscamos los máximos y mínimos 
  y_true_max = np.max(y_true)
  y_true_min = np.min(y_true)

  y_pred_max = np.max(y_pred)
  y_pred_min = np.min(y_pred)

  # Pos z
  plt.figure(figsize=(15, 6))
  plt.plot(y_true, label='Posiciones Z reales', linestyle='None', marker='.')
  plt.plot(y_pred, label='Posiciones Z predichas', linestyle = 'None',marker='o')
  # Dibujamos los max y min
  plt.axhline(y = y_true_max, color = 'red', linestyle = '-.', label=f'Máx_true: {y_true_max:.3f}')
  plt.axhline(y = y_pred_max, color = 'red', linestyle = ':', label= f'Máx_pred: {y_pred_max:.3f}')
  plt.axhline(y = y_true_min, color = 'blue', linestyle ='-.', label=f'Mín_true: {y_true_min:.3f}')
  plt.axhline(y = y_pred_min, color = 'blue', linestyle = ':',label= f'Mín_pred:{y_pred_min: .3f}')

  # plt.ylim(-35,-50) ##(-60,-30)
  plt.title('Comparación de Posiciones Z')
  plt.legend()
  plt.grid(True)
  plt.show()
graf_1D(y_true,y_pred)

In [ ]:
save_model = True

if save_model == True:
  model_z.save('../modelos_entrenamiento/2026/mod1_Z.keras')